# DoorDash Delivery ETA — Auditoria Aprofundada de Qualidade dos Dados

Este notebook realiza a auditoria aprofundada do dataset `historical_data.csv` (StrataScratch — DoorDash Delivery Duration Prediction), investigando:
1. Variável-alvo (`actual_delivery_time - created_at`) e outliers extremos
2. Valores negativos (`min_item_price`, `total_onshift_dashers`, `total_busy_dashers`, `total_outstanding_orders`)
3. Padrões de valores ausentes
4. Indicadores operacionais (`total_busy_dashers > total_onshift_dashers`)
5. Visualizações diagnósticas

> **Regras desta etapa:** O arquivo original `data/raw/historical_data.csv` permanece inalterado. Nenhum registro é removido, nenhum valor é imputado ou substituído, e nenhum modelo é treinado.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW_PATH = os.path.join("..", "data", "raw", "historical_data.csv")
FIG_DIR = os.path.join("..", "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Leitura somente leitura e criação de cópia de trabalho
raw_df = pd.read_csv(RAW_PATH)
df = raw_df.copy()

# Conversão de datas na cópia
df["created_at_dt"] = pd.to_datetime(df["created_at"], errors="coerce")
df["actual_delivery_time_dt"] = pd.to_datetime(df["actual_delivery_time"], errors="coerce")
df["duration_sec"] = (df["actual_delivery_time_dt"] - df["created_at_dt"]).dt.total_seconds()
df["duration_min"] = df["duration_sec"] / 60.0
df["duration_hours"] = df["duration_sec"] / 3600.0
df["created_hour"] = df["created_at_dt"].dt.hour
df["created_date"] = df["created_at_dt"].dt.date
df["created_dayofweek"] = df["created_at_dt"].dt.day_name()

print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas (originais: {raw_df.shape[1]})")

: 

## 1. Investigação da Variável-Alvo (`actual_delivery_time - created_at`)

In [ ]:
valid_dur = df["duration_sec"].dropna()
print(f"Registros válidos: {len(valid_dur)} | Nulos: {df['duration_sec'].isna().sum()}")

display(pd.DataFrame({
    "segundos": valid_dur.describe(),
    "minutos": df["duration_min"].dropna().describe()
}))

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
pct_df = pd.DataFrame({
    "percentil": [f"P{p*100:g}" for p in percentiles],
    "segundos": [valid_dur.quantile(p) for p in percentiles],
    "minutos": [df["duration_min"].dropna().quantile(p) for p in percentiles],
})
display(pct_df)

thresholds_min = [60, 90, 120, 180, 360, 720, 1440]
thresh_df = pd.DataFrame([
    {
        "limite": f"> {t} min ({t/60:.0f}h)" if t >= 360 else f"> {t} min",
        "quantidade": int((df["duration_min"] > t).sum()),
        "percentual_%": round((df["duration_min"] > t).sum() / len(valid_dur) * 100, 4)
    }
    for t in thresholds_min
])
display(thresh_df)

In [ ]:
# Top 30 maiores tempos de entrega e foco nos 3 registros > 24h
top30_cols = [
    "created_at", "actual_delivery_time", "duration_sec", "duration_min", "duration_hours",
    "market_id", "store_id", "store_primary_category", "order_protocol",
    "total_items", "subtotal", "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders"
]
top30 = df.sort_values("duration_sec", ascending=False).head(30)
display(top30[top30_cols])

# Padrões de outliers > 2h (120 min) por mercado e horário
outliers_2h = df[df["duration_min"] > 120]
print(f"Total > 2h: {len(outliers_2h)}")
mkt_summary = pd.DataFrame({
    "total_mercado": df["market_id"].value_counts(dropna=False),
    "gt_2h": outliers_2h["market_id"].value_counts(dropna=False),
})
mkt_summary["taxa_gt_2h_%"] = (mkt_summary["gt_2h"] / mkt_summary["total_mercado"] * 100).round(3)
display(mkt_summary)

hour_summary = pd.DataFrame({
    "total_hora": df["created_hour"].value_counts().sort_index(),
    "gt_2h": outliers_2h["created_hour"].value_counts().sort_index(),
})
hour_summary["taxa_gt_2h_%"] = (hour_summary["gt_2h"] / hour_summary["total_hora"] * 100).round(3)
display(hour_summary)

## 2. Investigação de Valores Negativos

In [ ]:
neg_cols = ["min_item_price", "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders"]

for col in neg_cols:
    sub = df[df[col] < 0]
    print(f"=== {col}: {len(sub)} registros negativos ({len(sub)/len(df)*100:.4f}%) ===")
    display(sub[col].value_counts().sort_index().to_frame("contagem"))
    display(sub[["created_at", "market_id", "store_id", "subtotal", "min_item_price",
                 "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders", "duration_min"]].head(15))

## 3. Investigação de Valores Ausentes

In [ ]:
missing_summary = pd.DataFrame({
    "ausentes": df[raw_df.columns].isna().sum(),
    "percentual_%": (df[raw_df.columns].isna().sum() / len(df) * 100).round(4)
}).sort_values("ausentes", ascending=False)
display(missing_summary[missing_summary["ausentes"] > 0])

op_cols = ["total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders"]
print("Padrão de nulidade simultânea nas 3 colunas operacionais:")
display(df[op_cols].isna().value_counts().to_frame("contagem"))

df["op_missing"] = df[op_cols].isna().any(axis=1)
print("Ausência operacional por market_id (%):")
display((pd.crosstab(df["market_id"].fillna(-1), df["op_missing"], normalize="index") * 100).round(2))

print("Comparação da duração (min) com vs sem ausência operacional:")
display(df.groupby("op_missing")["duration_min"].describe())

## 4. Investigação de Indicadores Operacionais (`total_busy_dashers > total_onshift_dashers`)

In [ ]:
both_valid = df[df["total_busy_dashers"].notna() & df["total_onshift_dashers"].notna()].copy()
busy_gt = both_valid[both_valid["total_busy_dashers"] > both_valid["total_onshift_dashers"]].copy()
busy_gt["diff"] = busy_gt["total_busy_dashers"] - busy_gt["total_onshift_dashers"]

print(f"Registros com ambos válidos: {len(both_valid)}")
print(f"total_busy_dashers > total_onshift_dashers: {len(busy_gt)} "
      f"({len(busy_gt)/len(df)*100:.2f}% do total | {len(busy_gt)/len(both_valid)*100:.2f}% dos válidos)")

display(busy_gt["diff"].describe().to_frame("busy - onshift"))
display(busy_gt[["created_at", "market_id", "store_id", "total_onshift_dashers",
                 "total_busy_dashers", "diff", "total_outstanding_orders", "duration_min"]].head(10))

## 5. Visualizações

In [ ]:
p99_min = df["duration_min"].quantile(0.99)

# 5.1 Distribuição da variável-alvo (até P99)
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df.loc[df["duration_min"] <= p99_min, "duration_min"], bins=60, kde=True, ax=ax, color="#2b5c8f")
ax.set_title(f"Distribuição da Duração de Entrega (até P99 = {p99_min:.1f} min)")
ax.set_xlabel("Duração (minutos)")
ax.set_ylabel("Frequência")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "target_distribution.png"), dpi=150)
plt.show()

# 5.2 Distribuição em escala log10
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(np.log10(df["duration_sec"].dropna()), bins=60, kde=True, ax=ax, color="#1f77b4")
ax.set_title("Distribuição da Duração de Entrega (Escala Log10 em Segundos)")
ax.set_xlabel("log10(Duração em segundos)")
ax.set_ylabel("Frequência")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "target_distribution_log.png"), dpi=150)
plt.show()

# 5.3 Boxplot da duração
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df["duration_min"].dropna(), ax=axes[0], color="#4c72b0")
axes[0].set_yscale("log")
axes[0].set_title("Boxplot da Duração (Escala Log)")
axes[0].set_ylabel("Duração (minutos, log)")
sns.boxplot(y=df.loc[df["duration_min"] <= 180, "duration_min"], ax=axes[1], color="#55a868")
axes[1].set_title("Boxplot da Duração (<= 180 min)")
axes[1].set_ylabel("Duração (minutos)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "target_boxplot.png"), dpi=150)
plt.show()

# 5.4 Duração por faixa horária
fig, ax = plt.subplots(figsize=(12, 5))
order_hours = sorted(df["created_hour"].dropna().unique())
sns.boxplot(data=df[df["duration_min"] <= p99_min], x="created_hour", y="duration_min",
            order=order_hours, ax=ax, showfliers=False, color="#4c72b0")
ax.set_title("Duração da Entrega por Hora de Criação (sem outliers extremos)")
ax.set_xlabel("Hora (created_at)")
ax.set_ylabel("Duração (minutos)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "duration_by_hour.png"), dpi=150)
plt.show()

# 5.5 Duração por dia da semana
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df[df["duration_min"] <= p99_min], x="created_dayofweek", y="duration_min",
            order=day_order, ax=ax, showfliers=False, color="#8172b3")
ax.set_title("Duração da Entrega por Dia da Semana (sem outliers extremos)")
ax.set_xlabel("Dia da Semana")
ax.set_ylabel("Duração (minutos)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "duration_by_dayofweek.png"), dpi=150)
plt.show()

# 5.6 Distribuição dos valores negativos
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, neg_cols):
    neg_vals = df.loc[df[col] < 0, col]
    vc = neg_vals.value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color="#c44e52")
    ax.set_title(f"Valores Negativos: {col} (n={len(neg_vals)})")
    ax.set_xlabel("Valor")
    ax.set_ylabel("Contagem")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "negative_values_distribution.png"), dpi=150)
plt.show()

# 5.7 Ausência de dados por variável
miss_plot = missing_summary[missing_summary["ausentes"] > 0].sort_values("percentual_%", ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(miss_plot.index, miss_plot["percentual_%"], color="#dd8452")
for i, (cnt, pct) in enumerate(zip(miss_plot["ausentes"], miss_plot["percentual_%"])):
    ax.text(pct + 0.1, i, f"{cnt} ({pct:.2f}%)", va="center")
ax.set_title("Percentual de Valores Ausentes por Variável")
ax.set_xlabel("% Ausente")
ax.set_xlim(0, miss_plot["percentual_%"].max() * 1.25)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "missing_values_by_variable.png"), dpi=150)
plt.show()